# Workflow

1. res_init
2. res_init2table
3. get_coordination
4. get_rxn
5. get_data

## `res_init.py`

In [3]:
import pandas as pd
from pathlib import Path
import sys
import os

os.makedirs('tsv', exist_ok=True)
os.makedirs('img', exist_ok=True)


# === Define input residue data ===
# Each tuple contains: (category label, residue name, residue index or list of indices)
residue_data = [
    ("metal", "MG", "371"),
    ("dna3term", "DA", "13"),
    ("dna5term", "DG", "14"),
    ("nuc", "WAT", "729"),
    ("base", "HIS", "303"),
    ("rescoord", "ASN", "324"),
    ("watcoord", "WAT", "601,611,612")
]


# === Create initial DataFrame ===
columns = ["category", "resname", "residx"]
df = pd.DataFrame(residue_data, columns=columns)

# === Expand comma-separated residue indices into individual rows ===
df_expanded = df.assign(residx=df["residx"].str.split(',')).explode("residx")
df_expanded["residx"] = df_expanded["residx"].str.strip()

# === Derive filename based on script name ===
script_name = Path(__file__).stem if '__file__' in globals() else "res_init"
outfile = Path(f"tsv/{script_name}.tsv")
df_expanded.to_csv(outfile, sep='\t', index=False)

print(f"Residue definition file saved as: {outfile.resolve()}")
df

Residue definition file saved as: /archive/projects/ippoi/qmmm/old/wildtype/distances/tsv/res_init.tsv


,category,resname,residx
0,metal,MG,371
1,dna3term,DA,13
2,dna5term,DG,14
3,nuc,WAT,729
4,base,HIS,303
5,rescoord,ASN,324
6,watcoord,WAT,"601,611,612"


## `res_init2table.py`

In [6]:
import pandas as pd
from pathlib import Path
import pytraj as pt

# === Config ===
pdb_path = Path("../input/prod1.pdb")
residue_file = Path("tsv/res_init.tsv")

# === Load topology ===
traj = pt.load(str(pdb_path))
top = traj.top

# === Build PDB serial number lookup ===
pdb_atom_serials = {}
for line in pdb_path.read_text().splitlines():
    if line.startswith("ATOM") or line.startswith("HETATM"):
        serial = int(line[6:11].strip())
        name = line[12:16].strip()
        resname = line[17:20].strip()
        resid = int(line[22:26].strip())
        key = (resname, resid, name)
        pdb_atom_serials[key] = serial

# === Load residue definition ===
df_input = pd.read_csv(residue_file, sep='\t')
df_input["residx"] = df_input["residx"].astype(int)

# === Collect atom records ===
records = []
for _, row in df_input.iterrows():
    category, resname, residx = row["category"], row["resname"], row["residx"]

    if residx >= top.n_residues:
        print(f"[Warning] Residue index {residx} out of range. Skipping.")
        continue

    for atom_idx in pt.select(f":{residx}", top):
        atom = top.atom(atom_idx)
        key = (atom.resname, atom.resid + 1, atom.name)
        pdb_serial = pdb_atom_serials.get(key, "NA")
        records.append({
            "category": category,
            "resname": atom.resname,
            "resid": atom.resid + 1,
            "atom_name": atom.name,
            "atom_idx": pdb_serial + 1,
            "mask": f":{atom.resid}@{atom.name}"
        })

# === Output as multi-index TSV ===
df_atoms = pd.DataFrame(records)
df_atoms.set_index(["category", "resname", "resid", "atom_name"], inplace=True)

outfile = Path("tsv/res_init2table.tsv")
df_atoms.to_csv(outfile, sep='\t')
print(f"Saved: {outfile.resolve()}")
df_atoms

Saved: /archive/projects/ippoi/qmmm/old/wildtype/distances/tsv/res_init2table.tsv


atom_idx     mask
category resname resid atom_name                   
metal    MG      371   MG             6230  :370@MG
dna3term DA      13    P               379    :12@P
                       OP1             380  :12@OP1
                       OP2             381  :12@OP2
                       O5'             382  :12@O5'
...                                    ...      ...
watcoord WAT     611   H1            66899  :610@H1
                       H2            66900  :610@H2
                 612   O             66901   :611@O
                       H1            66902  :611@H1
                       H2            66903  :611@H2

[109 rows x 2 columns]

## `get_coordination.py`

In [7]:
import pandas as pd
import os
from pathlib import Path
import sys

# === Input configuration ===
table_path = Path("tsv/res_init2table.tsv")   # Atom metadata table
metal_mask = ":371@MG"                         # Fixed metal atom selection (e.g., Mg)

# === Load the TSV file into a MultiIndexed DataFrame ===
df = pd.read_csv(table_path, sep='\t', index_col=[0, 1, 2, 3])
idx = df.index  # MultiIndex: category, resname, resid, atom_name


# === Define filters for specific coordination logic ===

# 1. Keep all entries from 'coord' category with oxygen atoms, excluding 'nuc', 'base', and 'O' atom in rescoord
is_coord = idx.get_level_values("category").str.contains("coord", case=False)
is_oxygen = idx.get_level_values("atom_name").str.startswith("O")
is_not_nuc = ~idx.get_level_values("category").str.contains("nuc", case=False)
is_not_base = ~idx.get_level_values("category").str.contains("base", case=False)
is_not_rescoord_O = ~(
    (idx.get_level_values("category") == "rescoord") &
    (idx.get_level_values("atom_name") == "O")
)
coord_filter = is_coord & is_oxygen & is_not_nuc & is_not_base & is_not_rescoord_O

# 2. Keep only "O3'" from dna3term
dna3_filter = (
    idx.get_level_values("category") == "dna3term"
) & (
    idx.get_level_values("atom_name") == "O3'"
)

# 3. Keep only "OP2" from dna5term
dna5_filter = (
    idx.get_level_values("category") == "dna5term"
) & (
    idx.get_level_values("atom_name") == "OP2"
)

# === Combine all valid filters ===
final_mask = coord_filter | dna3_filter | dna5_filter
filtered = df[final_mask]

# === Save AMBER mask output to a TSV file named after this script ===
def save_amber_mask(dataframe):
    try:
        script_name = os.path.splitext(os.path.basename(__file__))[0]
    except NameError:
        script_name = "get_coordination"
    output_filename = f"tsv/{script_name}.tsv"

    lines = [f"{metal_mask} :{resid}@{atom_name}" for _, _, resid, atom_name in dataframe.index]
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("Atoms Pairs\n")
        f.write("\n".join(lines))

save_amber_mask(filtered)

## `get_rxn.py`

In [8]:
import pandas as pd
import os
from pathlib import Path

# === Input configuration ===
table_path = Path("tsv/res_init2table.tsv")

# === Load the TSV file into a MultiIndexed DataFrame ===
df = pd.read_csv(table_path, sep='\t', index_col=[0, 1, 2, 3])
idx = df.index

# === Define filters for specific atoms by category ===
o3_filter = (idx.get_level_values("category") == "dna3term") & (idx.get_level_values("atom_name") == "O3'")
p_filter = (idx.get_level_values("category") == "dna5term") & (idx.get_level_values("atom_name") == "P")
o_filter = (idx.get_level_values("category") == "nuc") & (idx.get_level_values("atom_name") == "O")
h1_filter = (idx.get_level_values("category") == "nuc") & (idx.get_level_values("atom_name") == "H1")
h2_filter = (idx.get_level_values("category") == "nuc") & (idx.get_level_values("atom_name") == "H2")
nd1_filter = (idx.get_level_values("category") == "base") & (idx.get_level_values("atom_name") == "ND1")

# === Combine and extract relevant entries ===
final_mask = o3_filter | p_filter | o_filter | h1_filter | h2_filter | nd1_filter
atoms_df = df[final_mask]
atoms_idx = atoms_df.index

# === Organize atom lookups by atom name ===
def get_atoms(atom_name):
    return atoms_df.loc[atoms_idx.get_level_values("atom_name") == atom_name].index.tolist()

o3_atoms = get_atoms("O3'")
p_atoms = get_atoms("P")
o_atoms = get_atoms("O")
h1_atoms = get_atoms("H1")
h2_atoms = get_atoms("H2")
nd1_atoms = get_atoms("ND1")

# === Define desired interaction pairs ===
def pair_masks(atom_list1, atom_list2):
    pairs = []
    for _, _, resid1, atom1 in atom_list1:
        for _, _, resid2, atom2 in atom_list2:
            pairs.append(f":{resid1}@{atom1} :{resid2}@{atom2}")
    return pairs

# O3' to P
o3_p_pairs = pair_masks(o3_atoms, p_atoms)
# P to O
p_o_pairs = pair_masks(p_atoms, o_atoms)
# O - H1, O - H2
o_h1_pairs = pair_masks(o_atoms, h1_atoms)
o_h2_pairs = pair_masks(o_atoms, h2_atoms)
# H1 - ND1, H2 - ND1
h1_nd1_pairs = pair_masks(h1_atoms, nd1_atoms)
h2_nd1_pairs = pair_masks(h2_atoms, nd1_atoms)

# Combine all pairs
all_pairs = o3_p_pairs + p_o_pairs + o_h1_pairs + o_h2_pairs + h1_nd1_pairs + h2_nd1_pairs

# === Save to file ===
def save_rxn_mask(lines):
    try:
        script_name = os.path.splitext(os.path.basename(__file__))[0]
    except NameError:
        script_name = "get_rxn"
    output_filename = f"tsv/{script_name}.tsv"
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("Atom Pairs\n")
        f.write("\n".join(lines))

save_rxn_mask(all_pairs)
print(all_pairs)

[":13@O3' :14@P", ':14@P :729@O', ':729@O :729@H1', ':729@O :729@H2', ':729@H1 :303@ND1', ':729@H2 :303@ND1']


## `get_data.py`

In [10]:
import pytraj as pt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from multiprocessing import Pool
import os

# === Config ===
parm = "../input/step3_pbcsetup.parm7"
base_dir = "../"
reaction_coords = np.round(np.arange(-1.90, 2.21, 0.10), 2)

# === Validate and Map Windows ===
all_dirs = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and d.isdigit()])
if len(all_dirs) != len(reaction_coords):
    raise ValueError("Mismatch between detected windows and reaction coordinate steps.")
window_to_rc = dict(zip(all_dirs, reaction_coords))

# === Load TSV Inputs ===
coord_df = pd.read_csv("get_coordination.tsv", sep='\t')
rxn_df = pd.read_csv("get_rxn.tsv", sep='\t')

# === Core Distance Computation Function ===
def process_window(window_name):
    rc = window_to_rc[window_name]
    window_path = os.path.join(base_dir, window_name)
    traj_files = sorted(glob(os.path.join(window_path, "step6.0*_equilibration.nc")))
    if not traj_files:
        return [], [], []

    traj = pt.iterload(traj_files, parm)

    # Coordination
    coord_data = []
    for mask in coord_df['Atoms Pairs']:
        distances = pt.distance(traj, mask=mask)
        coord_data.append([rc, mask, np.mean(distances), np.std(distances)])

    # Reaction Path
    rxn_data = []
    rxn_raw = []
    for mask in rxn_df['Atom Pairs']:
        distances = pt.distance(traj, mask=mask)
        rxn_data.append([rc, mask, np.mean(distances), np.std(distances)])
        rxn_raw.extend([[rc, mask, d] for d in distances])

    return coord_data, rxn_data, rxn_raw

# === Run in Parallel (2 Processes) ===
with Pool(processes=2) as pool:
    results = pool.map(process_window, all_dirs)

# === Merge Results ===
coord_results = [row for result in results for row in result[0]]
rxn_results = [row for result in results for row in result[1]]
rxn_raw_data = [row for result in results for row in result[2]]

# === Export TSVs ===
coord_df_out = pd.DataFrame(coord_results, columns=["Reaction Coordinate", "Atoms Pairs", "Average Distance", "Std Dev"])
coord_df_out.to_csv("tsv/distances_results.tsv", sep='\t', index=False)

rxn_df_out = pd.DataFrame(rxn_results, columns=["Reaction Coordinate", "Atom Pairs", "Average Distance", "Std Dev"])
rxn_df_out.to_csv("tsv/reaction_distances.tsv", sep='\t', index=False)

raw_df = pd.DataFrame(rxn_raw_data, columns=["Reaction Coordinate", "Atom Pairs", "Distance"])
raw_df.to_csv("tsv/reaction_raw_distances.tsv", sep='\t', index=False)

# === Plotting Function ===
def plot_distances(df, title, file_name, colname="Atoms Pairs"):
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.set_xlim(-2.0, 2.5)
    unique_pairs = df[colname].unique()
    lines, labels = [], []

    for pair in unique_pairs:
        subset = df[df[colname] == pair].sort_values("Reaction Coordinate")
        rc = subset["Reaction Coordinate"].values
        avg = subset["Average Distance"].values
        std = subset["Std Dev"].values
        label = pair.replace(":", " ").replace("@", "-")
        line, = ax.plot(rc, avg, label=label)
        ax.fill_between(rc, avg - std, avg + std, alpha=0.2, color=line.get_color())
        lines.append(line)
        labels.append(label)

    ax.set_xlabel("Reaction Coordinate")
    ax.set_ylabel("Distance (Angstroms)")
    ax.set_title(title)
    ax.grid(True)
    ax.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), fontsize='small', ncol=2, frameon=True)
    plt.tight_layout(rect=[0, 0.1, 1, 1])
    plt.savefig(file_name)
    plt.close()

# === Generate Plots ===
plot_distances(coord_df_out, "Coordination Distances", "img/data_coordination_plot_all_pairs.png")
plot_distances(rxn_df_out, "Reaction Path Distances", "img/data_reaction_plot_all_pairs.png", colname="Atom Pairs")